
# CFPB Consumer Complaint Classification

## 01 — Data Understanding

### Objective
Explore the CFPB consumer complaint dataset before preprocessing and model training.

We will investigate:
- Dataset shape and columns
- Missing values
- Product class distribution
- Duplicate narratives
- Conflicting labels

**Input (X):** Consumer complaint narrative

**Target (y):** Product


In [ ]:

from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/CCDB_Export_16_March_2026.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

missing.sort_values("Missing Percentage", ascending=False)


## 02 — Initial Data Cleaning

The dataset contains a large proportion of missing complaint narratives.

Since our model uses complaint text as input, rows without narratives cannot be used for supervised text classification.

We will remove these rows from a working copy while preserving the original dataset.


In [ ]:

df_clean = df.dropna(
    subset=["Consumer complaint narrative"]
).copy()

print("Original rows:", len(df))
print("Rows after removing missing narratives:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))



### Conflicting Labels Analysis

Some identical complaint narratives are associated with different product categories.

Since our classifier uses only the complaint narrative as input, these conflicting labels introduce ambiguity.

We will identify and inspect these cases before deciding how to handle them.


In [ ]:

TEXT_COL = "Consumer complaint narrative"
TARGET_COL = "Product"

label_counts = df_clean.groupby(TEXT_COL)[TARGET_COL].nunique()

conflicting_texts = label_counts[label_counts > 1].index

print("Conflicting narratives:", len(conflicting_texts))

print(
    "Rows affected:",
    df_clean[TEXT_COL].isin(conflicting_texts).sum()
)



### Removing Conflicting Narratives

We exclude narratives associated with multiple product labels because our model relies exclusively on complaint text.

This avoids ambiguous training targets while preserving the original dataset for future analysis.


In [ ]:

df_clean = df_clean[
    ~df_clean[TEXT_COL].isin(conflicting_texts)
].copy()

print("Remaining rows:", len(df_clean))



### Removing Exact Duplicate Narratives

Identical complaint narratives may appear multiple times in the dataset.

We remove exact duplicates to prevent identical texts from appearing in both training and test sets.

This helps produce a more reliable evaluation on previously unseen complaint texts.


In [ ]:

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset=[TEXT_COL],
    keep="first"
).copy()

df_clean = df_clean.reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after:", len(df_clean))
print("Duplicates removed:", rows_before - len(df_clean))
print("Remaining duplicate narratives:", df_clean[TEXT_COL].duplicated().sum())


In [ ]:
class_counts = df_clean[TARGET_COL].value_counts()

print("Total tickets:", len(df_clean))
print("Number of classes:", df_clean[TARGET_COL].nunique())

class_counts


### Product Class Distribution

We examine the distribution of complaint categories after initial cleaning.

An imbalanced dataset can cause the classifier to perform better on majority classes while struggling with minority classes.

We will evaluate the model using macro-F1 in addition to accuracy.


In [ ]:

import matplotlib.pyplot as plt

class_counts = df_clean[TARGET_COL].value_counts()

plt.figure(figsize=(12, 6))

class_counts.sort_values().plot(kind="barh")

plt.title("Product Class Distribution After Cleaning")
plt.xlabel("Number of Complaints")
plt.ylabel("Product Category")

plt.tight_layout()
plt.show()



### Complaint Text Length Analysis

We analyze the length of complaint narratives to understand how much text each sample contains.

This helps us identify unusually short or long complaints and informs future preprocessing and model configuration decisions.


In [ ]:

df_clean["text_length"] = df_clean[TEXT_COL].str.len()

df_clean["word_count"] = df_clean[TEXT_COL].str.split().str.len()

df_clean[["text_length", "word_count"]].describe()



### Complaint Word Count Distribution

We visualize the distribution of complaint lengths to identify skewness and unusually long narratives.

Long complaints may influence processing time and future Transformer tokenization decisions.

We will inspect these observations before deciding whether any length-based filtering is necessary.


In [ ]:

plt.figure(figsize=(10, 5))

plt.hist(df_clean["word_count"], bins=50)

plt.title("Distribution of Complaint Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Number of Complaints")

plt.tight_layout()
plt.show()


In [ ]:

longest_tickets = df_clean.nlargest(5, "word_count")

longest_tickets[
    [TARGET_COL, "word_count", "text_length"]
]


In [ ]:

for _, row in longest_tickets.head(3).iterrows():
    print("=" * 60)
    print("Product:", row[TARGET_COL])
    print("Word count:", row["word_count"])
    print("Narrative preview:")
    print(row[TEXT_COL][:500])
    print()


In [ ]:

def type_token_ratio(text):
    words = text.lower().split()

    if not words:
        return 0

    return len(set(words)) / len(words)


longest_tickets = longest_tickets.copy()

longest_tickets["ttr"] = longest_tickets[TEXT_COL].apply(
    type_token_ratio
)

longest_tickets[
    [TARGET_COL, "word_count", "ttr"]
]


In [ ]:
lowest_ttr_ticket = longest_tickets.loc[
    longest_tickets["ttr"].idxmin()
]

text = lowest_ttr_ticket[TEXT_COL]

print("Product:", lowest_ttr_ticket[TARGET_COL])
print("Word count:", lowest_ttr_ticket["word_count"])
print("TTR:", round(lowest_ttr_ticket["ttr"], 3))

print("\nBEGINNING:")
print(text[:400])

print("\nMIDDLE:")
middle = len(text) // 2
print(text[middle:middle + 400])

print("\nEND:")
print(text[-400:])


### Redaction Marker Analysis

Some complaint narratives contain repeated redaction markers such as XXXX.

We investigate how frequently these markers appear before deciding whether they require text preprocessing.

We preserve the original complaint narratives and avoid removing meaningful financial information.


In [ ]:

df_clean["redaction_count"] = (
    df_clean[TEXT_COL]
    .str.count(r"\bXXXX\b", flags=0)
)

print(
    "Tickets containing XXXX:",
    (df_clean["redaction_count"] > 0).sum()
)

print(
    "Percentage:",
    round(
        (df_clean["redaction_count"] > 0).mean() * 100,
        2
    )
)

print(
    "Maximum XXXX occurrences:",
    df_clean["redaction_count"].max()
)



### Redaction Severity Analysis

We measure the proportion of redaction markers in each complaint.

This helps identify narratives dominated by anonymization placeholders rather than meaningful complaint text.

No rows will be removed at this stage.


In [ ]:

df_clean["redaction_ratio"] = (
    df_clean["redaction_count"] /
    df_clean["word_count"]
)

print("Average redaction ratio:",
      round(df_clean["redaction_ratio"].mean(), 3))

print("Tickets with over 20% redaction:",
      (df_clean["redaction_ratio"] > 0.20).sum())

print("Tickets with over 50% redaction:",
      (df_clean["redaction_ratio"] > 0.50).sum())

print("Maximum redaction ratio:",
      round(df_clean["redaction_ratio"].max(), 3))


In [ ]:

import re

high_redaction = df_clean[
    df_clean["redaction_ratio"] > 0.50
].copy()

def count_words_without_redactions(text):
    cleaned = re.sub(r"\bXXXX\b", " ", text)
    return len(cleaned.split())

high_redaction["remaining_words"] = (
    high_redaction[TEXT_COL].apply(
        count_words_without_redactions
    )
)

high_redaction[
    [TARGET_COL, "word_count", "redaction_count", "remaining_words"]
].sort_values("remaining_words")


In [ ]:

short_redacted = high_redaction[
    high_redaction["remaining_words"] < 15
]

for _, row in short_redacted.iterrows():
    text_without_redactions = re.sub(
        r"\bXXXX\b", " ", row[TEXT_COL]
    )

    print("=" * 50)
    print("Product:", row[TARGET_COL])
    print("Remaining words:", row["remaining_words"])
    print("Text:", " ".join(text_without_redactions.split()))



### Near-Duplicate Investigation

Exact duplicate narratives have already been removed.

However, complaints may share highly similar wording, particularly when consumers use legal templates.

We will investigate potential near-duplicates before deciding whether additional safeguards are needed for the train/test split.


In [ ]:

import re

def normalize_for_duplicate_check(text):
    text = text.lower()
    text = re.sub(r"\bxxxx\b", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_clean["normalized_text"] = df_clean[TEXT_COL].apply(
    normalize_for_duplicate_check
)

print(
    "Exact duplicates after normalization:",
    df_clean["normalized_text"].duplicated().sum()
)

print(
    "Empty texts after normalization:",
    df_clean["normalized_text"].eq("").sum()
)



### Label Consistency After Text Normalization

Some narratives become identical after case normalization, redaction-marker removal, and whitespace normalization.

We examine whether these normalized duplicates share the same product label.

This helps distinguish repeated training examples from ambiguous examples and informs our splitting strategy.


In [ ]:

normalized_groups = df_clean.groupby("normalized_text")

normalized_label_counts = normalized_groups[TARGET_COL].nunique()

normalized_conflicts = normalized_label_counts[
    normalized_label_counts > 1
]

print(
    "Normalized texts with conflicting labels:",
    len(normalized_conflicts)
)

print(
    "Rows affected:",
    df_clean["normalized_text"].isin(
        normalized_conflicts.index
    ).sum()
)

print(
    "Normalized duplicate rows:",
    df_clean["normalized_text"].duplicated().sum()
)



### Removing Normalized Duplicate Narratives

After normalization, 112 duplicate rows were identified, with no conflicting product labels.

For the initial baseline, we keep one example per normalized narrative to reduce potential train/test leakage.

The original raw dataset remains unchanged.


In [ ]:

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

df_clean = df_clean.reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after:", len(df_clean))
print("Normalized duplicates removed:", rows_before - len(df_clean))
print(
    "Remaining normalized duplicates:",
    df_clean["normalized_text"].duplicated().sum()
)



## 03 — Final Dataset Validation

After removing missing narratives, conflicting labels, and duplicate texts, we validate the final working dataset.

We check:
- Total number of samples
- Number of product categories
- Missing inputs and targets
- Remaining duplicates
- Class distribution

The original raw dataset remains unchanged.


In [ ]:

print("Final samples:", len(df_clean))
print("Number of classes:", df_clean[TARGET_COL].nunique())

print("\nMissing text:", df_clean[TEXT_COL].isna().sum())
print("Missing labels:", df_clean[TARGET_COL].isna().sum())

print(
    "Normalized duplicates:",
    df_clean["normalized_text"].duplicated().sum()
)

print("\nFinal class distribution:")
print(df_clean[TARGET_COL].value_counts())
